# Get started with Kumo Relational

Kumo Relational is a foundation model for relational data: it predicts from linked tables, such as customers, products and orders, with no feature engineering and no model trained for each task.
For linked tables, you describe what to predict as a short predictive query, and the model answers it from past examples in your own tables.

This notebook deploys the model in your workspace, then predicts which customers will stop ordering and who will place a large order, fills in a product attribute, and asks a large-order question through a metric view.
Each prediction is checked against what actually happened in Microsoft's AdventureWorks sample sales data, and compared with a simple rule, such as ranking customers by how long ago they last ordered.

## How Kumo Relational works

> **1. Enterprise data is a graph.**

Tables are connected by keys: here each order belongs to a customer, and each order line links a customer to a product.
These links give the model the context around every row, such as a customer's past orders.

> **2. Timestamps place events on a timeline.**

A table whose rows happen at a point in time, such as orders, is an event table.
You can pick any moment, the prediction time, and predict what happens next from what came before it.

> **3. Machine learning tasks can be written as predictive queries.**

A predictive query says what to predict, and for which entity, such as each customer or each product.
If you know SQL, the Predictive Query Language (PQL) will feel familiar:

```
PREDICT COUNT(sales_order_headers.*, 0, 90, days) = 0   ← no orders in the next 90 days
FOR EACH customers.CustomerID                           ← one prediction per customer
```

There is no training step: for each request, the client reads past cases with known outcomes from your tables, called context examples, and the model predicts from them.

## Before you start

You need:

- the Kumo Relational listing installed from Databricks Marketplace;
- permission to create a Model Serving endpoint, in a region with GPU Model Serving;
- a catalog and schema where you can create tables;
- serverless compute with environment 4 or later, or Databricks Runtime 17.3 LTS or later.

The first deployment of the endpoint takes 20 to 40 minutes, then the rest of the notebook runs in a few minutes.
The endpoint bills for GPU time while it runs; it scales to zero when idle, and the last step shows how to delete it.

Run each cell in order with Shift+Enter.

**Let's get started!**

## Step 1. Install the client

Install the client, the `structured-data-models-client` package, with its Databricks extras, then restart Python.
pip may print a red message about `protobuf` versions; it does not affect this notebook.

In [ ]:
%pip install --quiet "structured-data-models-client[relational,databricks,databricks-serving]"
dbutils.library.restartPython()

## Step 2. Set your parameters

Set these names, then run the cell.
`MODEL_NAME` is the model from the listing, in the catalog you installed it into, and `CATALOG` and `SCHEMA` are where the sample tables go.

In [ ]:
import json
from datetime import timedelta

import pandas as pd
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    ServingModelWorkloadType,
)
from sdm_client import (
    KumoRelational,
    PredictiveQuery,
    Relationship,
    Schema,
    SDMClient,
)
from sdm_connectors import DatabricksSource, quote_ident

# The model from the listing, as <catalog>.<schema>.<model>, and its version.
MODEL_NAME = 'kumo_relational.models.kumo_relational'
MODEL_VERSION = '1'
ENDPOINT_NAME = 'kumo-relational'
# Where the sample tables go: a catalog and schema you can create tables in.
CATALOG = 'main'
SCHEMA = 'kumo_relational_quickstart'


def quoted(*parts):
    # A Unity Catalog name, each part quoted for SQL.
    return '.'.join(quote_ident(part, char='`') for part in parts)


def full_name(table):
    # The full name of a sample table.
    return quoted(CATALOG, SCHEMA, f'adventureworks_{table}')

## Step 3. Deploy the model

The next cell creates a Model Serving endpoint for the model in your workspace, on one GPU (`GPU_MEDIUM`, size Small) that scales to zero when idle.
It does not wait for the deployment; Step 6 does.
If the endpoint already exists, the cell checks that it serves this model.
You can also create it with **Serve this model** on the model's page in Catalog Explorer.

In [ ]:
workspace = WorkspaceClient()
try:
    endpoint = workspace.serving_endpoints.get(ENDPOINT_NAME)
except NotFound:
    workspace.serving_endpoints.create(
        name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(
            name=ENDPOINT_NAME,
            served_entities=[
                ServedEntityInput(
                    entity_name=MODEL_NAME,
                    entity_version=MODEL_VERSION,
                    workload_type=ServingModelWorkloadType.GPU_MEDIUM,
                    workload_size='Small',
                    scale_to_zero_enabled=True,
                )
            ],
        ),
    )
    print(
        f'Creating the endpoint {ENDPOINT_NAME}; this takes 20 to 40 minutes.'
    )
else:
    config = endpoint.config or endpoint.pending_config
    served = {
        (entity.entity_name.lower(), entity.entity_version)
        for entity in config.served_entities
    }
    if served != {(MODEL_NAME.lower(), MODEL_VERSION)}:
        raise ValueError(
            f'The endpoint {ENDPOINT_NAME} already exists and serves {sorted(served)}, '
            f'not {MODEL_NAME} version {MODEL_VERSION}. Set ENDPOINT_NAME to another name.'
        )
    print(f'The endpoint {ENDPOINT_NAME} already exists and serves this model.')

## Step 4. Load the sample data

The next cell downloads the AdventureWorks customers, products, order headers and order lines from the [Microsoft SQL Server samples repository](https://github.com/microsoft/sql-server-samples/tree/master/samples/databases/adventure-works/oltp-install-script) at a fixed commit, adds each order's customer and date to its lines, and saves the tables to your schema.
It needs internet access, and skips the download if the tables already exist with the expected columns.

> Each user is responsible for checking the content of datasets and the applicable licenses and determining if suitable for the intended use.
> The repository's samples are provided under the [MIT License](https://github.com/microsoft/sql-server-samples/blob/master/license.txt).

In [ ]:
URL = (
    'https://raw.githubusercontent.com/microsoft/sql-server-samples/'
    'b47eadc85277f72843647223cde53aa1b85ffcb8/samples/databases/'
    'adventure-works/oltp-install-script/{}.csv'
)
# Each table's file, and the file's columns in order.
FILES = {
    'customers': (
        'Customer',
        'CustomerID PersonID StoreID TerritoryID AccountNumber rowguid '
        'ModifiedDate',
    ),
    'products': (
        'Product',
        'ProductID Name ProductNumber MakeFlag FinishedGoodsFlag Color '
        'SafetyStockLevel ReorderPoint StandardCost ListPrice Size '
        'SizeUnitMeasureCode WeightUnitMeasureCode Weight DaysToManufacture '
        'ProductLine Class Style ProductSubcategoryID ProductModelID '
        'SellStartDate SellEndDate DiscontinuedDate rowguid ModifiedDate',
    ),
    'sales_order_headers': (
        'SalesOrderHeader',
        'SalesOrderID RevisionNumber OrderDate DueDate ShipDate Status '
        'OnlineOrderFlag SalesOrderNumber PurchaseOrderNumber AccountNumber '
        'CustomerID SalesPersonID TerritoryID BillToAddressID ShipToAddressID '
        'ShipMethodID CreditCardID CreditCardApprovalCode CurrencyRateID '
        'SubTotal TaxAmt Freight TotalDue Comment rowguid ModifiedDate',
    ),
    'sales_order_details': (
        'SalesOrderDetail',
        'SalesOrderID SalesOrderDetailID CarrierTrackingNumber OrderQty '
        'ProductID SpecialOfferID UnitPrice UnitPriceDiscount LineTotal '
        'rowguid ModifiedDate',
    ),
}
# The columns each saved table has: order lines also get CustomerID and OrderDate.
COLUMNS = {table: columns.split() for table, (_, columns) in FILES.items()}
COLUMNS['sales_order_details'] += ['CustomerID', 'OrderDate']

existing = [
    table for table in FILES if spark.catalog.tableExists(full_name(table))
]
for table in existing:
    if spark.table(full_name(table)).columns != COLUMNS[table]:
        raise ValueError(
            f'{full_name(table)} already exists with other columns. '
            'Drop it, or set SCHEMA to another schema.'
        )
if len(existing) == len(FILES):
    print(f'The sample tables are already in {CATALOG}.{SCHEMA}.')
else:
    frames = {}
    for table, (file, columns) in FILES.items():
        frame = pd.read_csv(
            URL.format(file),
            sep='\t',
            header=None,
            names=columns.split(),
            dtype_backend='numpy_nullable',
        )
        for column in frame.columns:
            if column.endswith('Date'):
                frame[column] = pd.to_datetime(frame[column])
        frames[table] = frame
    orders = frames['sales_order_headers'][
        ['SalesOrderID', 'CustomerID', 'OrderDate']
    ]
    frames['sales_order_details'] = frames['sales_order_details'].merge(
        orders, on='SalesOrderID', validate='many_to_one'
    )
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {quoted(CATALOG, SCHEMA)}')
    for table, frame in frames.items():
        spark.createDataFrame(frame).write.mode('ignore').saveAsTable(
            full_name(table)
        )
    print(f'Saved the sample tables to {CATALOG}.{SCHEMA}.')

## Step 5. Describe the tables

The model needs each table's primary key, how the tables link, and which tables are event tables.
`DatabricksSource` reads the tables in place, with your permissions, and sends the endpoint only the rows each prediction needs.
Order headers and order lines are event tables, with `OrderDate` as their time column; customers and products have none.

The cell also leaves out columns that would mislead the model: random IDs and free text, which the model does not read, and columns that can show what happened after the prediction time, such as edit times, product sell dates and ship dates.

In [ ]:
source = DatabricksSource(spark, catalog=CATALOG, schema=SCHEMA)
schema = Schema.from_source(
    source,
    {
        'customers': 'adventureworks_customers',
        'products': 'adventureworks_products',
        'sales_order_headers': 'adventureworks_sales_order_headers',
        'sales_order_details': 'adventureworks_sales_order_details',
    },
    primary_keys={
        'customers': 'CustomerID',
        'products': 'ProductID',
        'sales_order_headers': 'SalesOrderID',
        'sales_order_details': 'SalesOrderDetailID',
    },
    time_columns={
        'sales_order_headers': 'OrderDate',
        'sales_order_details': 'OrderDate',
        'customers': None,
        'products': None,
    },
    relationships=[
        Relationship(
            'sales_order_headers', 'CustomerID', 'customers', 'CustomerID'
        ),
        Relationship(
            'sales_order_details', 'CustomerID', 'customers', 'CustomerID'
        ),
        Relationship(
            'sales_order_details', 'ProductID', 'products', 'ProductID'
        ),
    ],
)
leave_out = {
    'customers': ['rowguid', 'ModifiedDate', 'AccountNumber'],
    'products': [
        'rowguid',
        'ModifiedDate',
        'Name',
        'ProductNumber',
        'SellStartDate',
        'SellEndDate',
        'DiscontinuedDate',
    ],
    'sales_order_headers': [
        'rowguid',
        'ModifiedDate',
        'SalesOrderNumber',
        'PurchaseOrderNumber',
        'AccountNumber',
        'CreditCardApprovalCode',
        'Comment',
        'RevisionNumber',
        'Status',
        'ShipDate',
    ],
    'sales_order_details': ['rowguid', 'ModifiedDate', 'CarrierTrackingNumber'],
}
for table, columns in leave_out.items():
    for column in columns:
        del schema[table][column]

print(*schema.tables.values(), sep='\n')
print(*schema.relationships, sep='\n')

## Step 6. Connect to the model

The next cell waits until the endpoint is ready, then connects to it.
If the endpoint has scaled to zero, the first prediction takes a few extra minutes while it starts.

In [ ]:
endpoint = workspace.serving_endpoints.wait_get_serving_endpoint_not_updating(
    ENDPOINT_NAME, timeout=timedelta(hours=1)
)
ready = endpoint.state.ready.value if endpoint.state.ready else 'unknown'
if ready != 'READY':
    raise RuntimeError(
        f'The endpoint {ENDPOINT_NAME} is {ready}. '
        'Open it under Serving to see why, then run this cell again.'
    )
# Allow 15 minutes per request, for an endpoint that has to scale up from zero.
client = SDMClient.for_databricks_serving(ENDPOINT_NAME, timeout=900)
model = KumoRelational()
print(f'Connected to {ENDPOINT_NAME}.')

## Step 7. Make predictions

Each prediction takes three calls:

1. `PredictiveQuery(text, schema)` writes the predictive query.
2. `materialize(indices=[...])` reads the rows the model needs for the given entity IDs, context examples included.
3. `client.predict(request, model=model)` sends the request, and `to_pandas()` returns the result.

**Note:** AdventureWorks is sample data, and the results below are for demonstration.
Benchmark Kumo Relational on data representative of your own use case before drawing conclusions about its performance.

#### Predictive queries in short

<details>
<summary><b>Click here for a short introduction to predictive queries</b></summary>

**Entities**

`FOR EACH <table>.<primary key>` names the entity, such as `FOR EACH customers.CustomerID`; you pass the IDs to predict for as `indices`, so one query works for any set of IDs.

**Targets**

- `PREDICT <table>.<column>` predicts a value of the entity itself, such as a missing attribute.
- `PREDICT <aggregation>(<table>.<column>, <start>, <end>, <unit>)` predicts an aggregate over a time window after the prediction time, such as `0, 30, days` for the next 30 days.

**Aggregations**

`COUNT`, `SUM`, `AVG`, `MIN` and `MAX`; `COUNT(<table>.*, ...)` counts rows, such as orders.

**Conditions**

A comparison after the target turns it into a yes-or-no question: `COUNT(sales_order_headers.*, 0, 90, days) = 0` asks whether a customer places no order in the next 90 days.
A `WHERE` inside the aggregation counts only some rows: `COUNT(sales_order_headers.* WHERE sales_order_headers.TotalDue > 1000, 0, 90, days)` counts only orders over $1,000.
</details>

### Example 1: Predict which customers will stop ordering

Which active customers will place no order in the next 90 days?

```
PREDICT COUNT(sales_order_headers.*, 0, 90, days) = 0   ← no orders in the next 90 days
FOR EACH customers.CustomerID                           ← one prediction per customer
```

The `= 0` makes it a yes-or-no question, answered with a probability.

To check the answers against what actually happened, the cell moves the prediction time into the past with `anchor_time`, here 180 days before the last order in the data.
The model then sees only the orders placed up to that time.
The cell asks for a random sample of 1,000 customers who ordered in the year before, and reads what each did in the 90 days after and in the year before; Examples 2 and 4 use the same customers and prediction time.

`num_context_samples` sets how many context examples the model reads: 1,000 by default, up to 10,000, and 5,000 for the customer questions here.

In [ ]:
latest = pd.Timestamp(
    spark.sql(
        f'SELECT max(OrderDate) FROM {full_name("sales_order_headers")}'
    ).first()[0]
)
PAST = latest - timedelta(days=180)
NEXT_90_DAYS = (
    f"OrderDate > '{PAST}' AND OrderDate <= '{PAST + timedelta(days=90)}'"
)
LAST_YEAR = (
    f"OrderDate <= '{PAST}' AND OrderDate > '{PAST - timedelta(days=365)}'"
)
# What each customer did in the 90 days after the prediction time, and in the
# year before it. SubTotal is an order's revenue, the sum of its line totals.
customers = (
    spark.sql(f"""
    SELECT CustomerID,
           count_if({NEXT_90_DAYS}) AS orders,
           count_if({NEXT_90_DAYS} AND TotalDue > 1000) AS large_orders,
           coalesce(sum(SubTotal) FILTER (WHERE {NEXT_90_DAYS}), 0) AS revenue,
           datediff('{PAST}', max(OrderDate) FILTER (WHERE OrderDate <= '{PAST}'))
               AS days_since_order,
           coalesce(sum(TotalDue) FILTER (WHERE {LAST_YEAR}), 0) AS spend_last_year,
           count_if({LAST_YEAR} AND TotalDue > 1000) AS large_orders_last_year
    FROM {full_name('sales_order_headers')}
    GROUP BY CustomerID
    HAVING count_if({LAST_YEAR}) > 0
    ORDER BY md5(cast(CustomerID AS string))
    LIMIT 1000
""")
    .toPandas()
    .set_index('CustomerID')
)


def by_group(score, happened):
    # Four equal groups by score, such as the predicted probability: the
    # group's average score, and how often the event actually happened.
    if len(score) < 4:
        raise ValueError('Four groups need at least four customers.')
    labels = ['lowest', 'low', 'high', 'highest']
    group = pd.qcut(score.rank(method='first'), 4, labels=labels).rename(
        'group'
    )
    frame = pd.DataFrame({'predicted': score, 'actual': happened.astype(float)})
    return frame.groupby(group, observed=True).mean().round(2).reset_index()


request = PredictiveQuery(
    'PREDICT COUNT(sales_order_headers.*, 0, 90, days) = 0 '
    'FOR EACH customers.CustomerID',
    schema,
).materialize(
    indices=customers.index.tolist(), anchor_time=PAST, num_context_samples=5000
)
churn = (
    client.predict(request, model=model).to_pandas().join(customers, on='id')
)
stopped = churn['orders'] == 0
display(by_group(churn['probability_True'], stopped))
# A quarter of the customers, rated riskiest by the model and by the rule.
top = len(churn) // 4
riskiest = churn['probability_True'].nlargest(top).index
longest_gap = churn['days_since_order'].nlargest(top).index
print(
    f'No order in the 90 days: {stopped[riskiest].mean():.0%} of the {top} '
    f'customers rated riskiest, {stopped[longest_gap].mean():.0%} of the {top} '
    f'longest since their last order, and {stopped.mean():.0%} of all '
    f'{len(churn):,}.'
)
shares = pd.DataFrame(
    {
        'Kumo Relational': by_group(
            churn['probability_True'], stopped
        ).set_index('group')['actual'],
        'Days since last order': by_group(
            churn['days_since_order'], stopped
        ).set_index('group')['actual'],
    }
)
shares.plot.bar(
    title='Share of customers who placed no order in the 90 days',
    xlabel='Quarter of the customers, ranked by each score',
    ylabel='Share with no order',
    ylim=(0, 1.25),
    yticks=[0, 0.25, 0.5, 0.75, 1],
    rot=0,
)

How to interpret the result:

1. Each row is a quarter of the 1,000 customers, grouped by `probability_True`, the predicted probability of placing no order.
2. `predicted` and `actual`: the group's average probability, and the share that placed no order.
3. The printed line compares the 250 customers rated riskiest with the 250 who had gone longest without an order, a common rule of thumb.
4. The chart shows the share that placed no order in each quarter, with the customers ranked by predicted probability and by days since their last order.

The group rated riskiest stopped ordering far more often than the group rated safest, and the 250 rated riskiest more often than the 250 the rule picks, so the probability can rank customers for retention work.

### Example 2: Predict who will place a large order

Which of the same customers will place an order over $1,000 in the next 90 days?

```
PREDICT COUNT(sales_order_headers.*
              WHERE sales_order_headers.TotalDue > 1000, 0, 90, days) > 0   ← any order over $1,000 in 90 days
FOR EACH customers.CustomerID                                               ← one prediction per customer
```

The `WHERE` inside the aggregation counts only large orders, and `> 0` asks whether there is at least one.

In [ ]:
request = PredictiveQuery(
    'PREDICT COUNT(sales_order_headers.* WHERE sales_order_headers.TotalDue > 1000, '
    '0, 90, days) > 0 FOR EACH customers.CustomerID',
    schema,
).materialize(
    indices=customers.index.tolist(), anchor_time=PAST, num_context_samples=5000
)
large = (
    client.predict(request, model=model).to_pandas().join(customers, on='id')
)
placed = large['large_orders'] > 0
display(by_group(large['probability_True'], placed))
likeliest = large['probability_True'].nlargest(top).index
most_last_year = large.sort_values(
    ['large_orders_last_year', 'spend_last_year'], ascending=False
).index[:top]
print(
    f'Of the {placed.sum()} customers who placed a large order, '
    f'{placed[likeliest].sum()} are among the {top} rated most likely, and '
    f'{placed[most_last_year].sum()} among the {top} with the most large '
    'orders last year.'
)

How to interpret the result:

1. Each row is a quarter of the customers, grouped by `probability_True`, the predicted probability of a large order.
2. `predicted` and `actual`: the group's average probability, and the share that placed one.
3. The printed line compares the 250 customers rated most likely with the 250 who placed the most large orders in the year before.

Large orders came almost only from the half rated most likely, and more of them from the 250 rated most likely than from the 250 the rule picks, so the probability can pick whom to contact for an upsell.

### Example 3: Fill in a missing product attribute

A query with no time window predicts a value of the entity itself, which fills in values that are missing.

```
PREDICT products.ProductLine   ← R road, M mountain, T touring or S standard
FOR EACH products.ProductID    ← one prediction per product
```

To check it, the cell predicts the product line of 60 random products whose line is known, and compares the predictions with the recorded values.
The model never sees a product's own recorded value.

In [ ]:
lines = spark.sql(f"""
    SELECT ProductID, Name AS product, trim(ProductLine) AS recorded
    FROM {full_name('products')}
    WHERE ProductLine IS NOT NULL
    ORDER BY md5(cast(ProductID AS string))
    LIMIT 60
""").toPandas()

request = PredictiveQuery(
    'PREDICT products.ProductLine FOR EACH products.ProductID', schema
).materialize(indices=lines['ProductID'].tolist())
product_lines = client.predict(request, model=model).to_pandas()

lines['predicted'] = lines['ProductID'].map(
    product_lines.set_index('id')['prediction'].str.strip()
)
most_common = spark.sql(f"""
    SELECT trim(ProductLine) FROM {full_name('products')}
    WHERE ProductLine IS NOT NULL
    GROUP BY 1
    ORDER BY count(*) DESC
    LIMIT 1
""").first()[0]
correct = (lines['predicted'] == lines['recorded']).sum()
guessed = (lines['recorded'] == most_common).sum()
print(
    f'{correct} of {len(lines)} product lines predicted correctly; always '
    f'guessing the most common line, {most_common}, gets {guessed} right.'
)
display(lines[lines['predicted'] != lines['recorded']])

How to interpret the result:

1. The printed line counts the correct predictions, and the right guesses of a rule that always picks the most common line.
2. The table lists the products the model got wrong.

Predicted values like these can fill gaps in a catalog, but review them before writing them back to a source system.

### Example 4: Predict from a metric view

A Unity Catalog metric view defines how tables join and which measures matter, such as revenue.
The client reads that definition, so you can ask the same kind of question as Example 2 without describing the tables yourself.
The next cell creates or replaces a metric view over the sample tables, `adventureworks_sales_metrics`.

In [ ]:
view_name = full_name('sales_metrics')
spark.sql(f"""
CREATE OR REPLACE VIEW {view_name}
WITH METRICS LANGUAGE YAML AS $$
version: 0.1
source: {json.dumps(full_name('sales_order_details'))}
joins:
  - name: customers
    source: {json.dumps(full_name('customers'))}
    on: source.CustomerID = customers.CustomerID
  - name: products
    source: {json.dumps(full_name('products'))}
    on: source.ProductID = products.ProductID
dimensions:
  - name: SalesOrderDetailID
    expr: source.SalesOrderDetailID
  - name: SalesOrderID
    expr: source.SalesOrderID
  - name: OrderDate
    expr: source.OrderDate
  - name: OrderQty
    expr: source.OrderQty
  - name: UnitPrice
    expr: source.UnitPrice
  - name: LineTotal
    expr: source.LineTotal
  - name: Discounted
    expr: source.UnitPriceDiscount > 0
  - name: TerritoryID
    expr: customers.TerritoryID
  - name: StoreID
    expr: customers.StoreID
  - name: Color
    expr: products.Color
  - name: ListPrice
    expr: products.ListPrice
  - name: ProductLine
    expr: products.ProductLine
  - name: ProductSubcategoryID
    expr: products.ProductSubcategoryID
measures:
  - name: total_quantity
    expr: SUM(source.OrderQty)
  - name: revenue
    expr: SUM(source.LineTotal)
$$
""")
print(f'Created the metric view {view_name}.')

`source.semantic_model` reads the view, and `Schema.from_source` turns it into a schema.
The cell prints the predictive-query target behind each measure.

In [ ]:
view_schema = Schema.from_source(source.semantic_model(view_name))
print(*view_schema.conversion_notes, sep='\n')

Which of the same customers will spend more than $1,000 in the next 90 days, by the view's `revenue` measure?

```
PREDICT SUM(adventureworks_sales_order_details.LineTotal, 0, 90, days) > 1000   ← revenue over $1,000 in 90 days
FOR EACH customers.CustomerID                                                   ← one prediction per customer
```

The cell compares the predictions with the same rule as Example 2.

In [ ]:
request = PredictiveQuery(
    'PREDICT SUM(adventureworks_sales_order_details.LineTotal, 0, 90, days) '
    '> 1000 FOR EACH customers.CustomerID',
    view_schema,
).materialize(
    indices=customers.index.tolist(), anchor_time=PAST, num_context_samples=5000
)
over_1000 = (
    client.predict(request, model=model).to_pandas().join(customers, on='id')
)
ordered = over_1000['revenue'] > 1000
display(by_group(over_1000['probability_True'], ordered))
likeliest = over_1000['probability_True'].nlargest(top).index
most_last_year = over_1000.sort_values(
    ['large_orders_last_year', 'spend_last_year'], ascending=False
).index[:top]
print(
    f'Of the {ordered.sum()} customers who spent more than $1,000, '
    f'{ordered[likeliest].sum()} are among the {top} rated most likely, and '
    f'{ordered[most_last_year].sum()} among the {top} with the most large '
    'orders last year.'
)

How to interpret the result:

1. Each row is a quarter of the customers, grouped by `probability_True`, the predicted probability of spending more than $1,000.
2. `predicted` and `actual`: the group's average probability, and the share that did.
3. The printed line compares the 250 customers rated most likely with the 250 who placed the most large orders in the year before.

This lets a team ask questions about the measures it already defines in metric views.

## Step 8. Try it yourself

The next cell asks the question of Example 1 again, for the same customers at the same prediction time, with a window of `DAYS` days that you can change.
It compares each group's average prediction with what actually happened.
Change `DAYS`, or uncomment the challenge line, and run it again.

In [ ]:
DAYS = 90
# Challenge: uncomment the line, run the cell again, and compare with the actuals.
# DAYS = 180  # no order in the next half year

request = PredictiveQuery(
    f'PREDICT COUNT(sales_order_headers.*, 0, {DAYS}, days) = 0 '
    'FOR EACH customers.CustomerID',
    schema,
).materialize(
    indices=customers.index.tolist(), anchor_time=PAST, num_context_samples=5000
)
result = client.predict(request, model=model).to_pandas()

orders = (
    spark.sql(f"""
    SELECT CustomerID,
           count_if(OrderDate > '{PAST}'
                    AND OrderDate <= '{PAST + timedelta(days=DAYS)}') AS orders
    FROM {full_name('sales_order_headers')}
    GROUP BY CustomerID
""")
    .toPandas()
    .set_index('CustomerID')['orders']
)
result['orders'] = result['id'].map(orders)
display(by_group(result['probability_True'], result['orders'] == 0))

## Step 9. Use your own data

For your own linked tables, describe them and write a predictive query:

```python
source = DatabricksSource(spark, catalog='<your catalog>', schema='<your schema>')
schema = Schema.from_source(
    source,
    {'users': '<users table>', 'orders': '<orders table>'},
    primary_keys={'users': 'user_id', 'orders': 'order_id'},
    time_columns={'orders': 'ordered_at', 'users': None},
    relationships=[Relationship('orders', 'user_id', 'users', 'user_id')],
)
request = PredictiveQuery(
    'PREDICT COUNT(orders.*, 0, 90, days) = 0 FOR EACH users.user_id', schema
).materialize(indices=[...], num_context_samples=5000)
result = client.predict(request, model=model).to_pandas()
```

| Question | Predictive query |
|---|---|
| Will this customer stop ordering? | `PREDICT COUNT(orders.*, 0, 90, days) = 0 FOR EACH users.user_id` |
| How much will this customer spend? | `PREDICT SUM(orders.amount, 0, 90, days) FOR EACH users.user_id` |
| How many units of this item will be ordered? | `PREDICT SUM(order_lines.quantity, 0, 30, days) FOR EACH items.item_id` |
| What is this item's missing category? | `PREDICT items.category FOR EACH items.item_id` |

For a single table, you can also send labelled rows and the rows to score without a predictive query; see [One table](https://github.com/NVIDIA/structured-data-models-client/blob/main/docs/get-started/quickstart.md#one-table) in the client's quickstart.

Tips:

- Leave out random row IDs, free text, and columns filled in after the prediction time, such as edit times and ship dates.
- To keep the predictions, save them as a table: `spark.createDataFrame(result).write.saveAsTable('<table>')`.

Learn more:

- [Predictive Query reference](https://docs.nvidia.com/sdgm/reference/pq-reference-overview)
- [Writing predictive queries](https://docs.nvidia.com/sdgm/rfm/writing-predictive-queries)

## Step 10. Clean up

The endpoint bills for GPU time only while it runs.
To delete it, uncomment the last line and run the cell.
The sample tables and the metric view stay in your schema until you drop them.

In [ ]:
client.close()
# workspace.serving_endpoints.delete(ENDPOINT_NAME)